In [4]:
import pandas as pd
import sqlite3
import os
import datetime
import re
from multidata import casematch, ingest, manifest, paths
from dotenv import load_dotenv
import json

load_dotenv(paths.ROOT / ".env")

DATA_DIR = paths.DATA_DIR / "audio"
MANIFEST_PATH = paths.MANIFEST_PATH
HF_TOKEN = os.getenv('HF_TOKEN')

DEFAULT_PROMPT = (
    "Clinical consultation transcript between a healthcare provider and a patient "
    "discussing medical symptoms, history, diagnosis, and treatment plan, followed "
    "by feedback between the provider and their instructor. Formal medical "
    "terminology is used throughout."
)

In [7]:

import whisperx, gc, torch
device="cpu"

In [8]:
model = whisperx.load_model("large-v3", device, compute_type="int8",
asr_options={"initial_prompt":DEFAULT_PROMPT})


2026-08-26 13:20:11 - whisperx.asr - INFO - No language specified, language will be detected for each audio file (increases inference time)
2026-08-26 13:20:11 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../opt/homebrew/Caskroom/miniforge/base/envs/md-speech/lib/python3.11/site-packages/whisperx/assets/pytorch_model.bin`


In [10]:


filename = '20260226-130604-v309309-16.mp4'
case_id = casematch.resolve(os.path.join(DATA_DIR, filename), manifest_path=MANIFEST_PATH)
case_id
camera_id = filename.split('.')[0].split('-')[-1]
audio_filename = f'{camera_id}.wav'
audio = whisperx.load_audio(os.path.join(DATA_DIR,case_id,audio_filename))
result = model.transcribe(audio, batch_size=8)

2026-08-26 13:30:26 - whisperx.asr - INFO - Detected language: en (0.83) in first 30s of audio


In [11]:
align_model, meta = whisperx.load_align_model(result["language"], device)
result = whisperx.align(result["segments"], align_model, meta, audio, device)

In [13]:
#10:51 (?) on cpu
# 1:34 (!) on mps
dia = whisperx.diarize.DiarizationPipeline(token=HF_TOKEN, device='mps')
diarize_segments = dia(os.path.join(DATA_DIR,case_id,audio_filename), max_speakers=4)                 # optionally min/max speakers
result = whisperx.assign_word_speakers(diarize_segments, result)

2026-08-26 13:52:55 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


In [14]:
out_path = os.path.join("../data/transcripts", f'{case_id}_whisperx.json')
with open(out_path, "w") as f:
    json.dump(result, f, indent=4)
print(f"Wrote {out_path} ")

Wrote ../data/transcripts/261456_whisperx.json 


In [39]:
from multidata.elan import build_eaf

In [40]:
build_eaf(out_path, f'../data/raw/{filename}', f'../data/elan/{case_id}.eaf')

File created successfully: ../data/elan/261498.eaf (684 words added)
Identified Tiers: ['default', 'SPEAKER_02', 'Unknown_Speaker', 'SPEAKER_03', 'SPEAKER_01', 'SPEAKER_00']


'../data/elan/261498.eaf'

In [41]:
from multidata.acoustics import extract_features
extract_features(os.path.join(DATA_DIR,audio_filename),
f'../data/praat/{case_id}.praat')

Data successfully exported to ../data/praat/261498.praat


'../data/praat/261498.praat'

In [15]:
from multidata.asr import transcribe_whisperx_disfluent

result = transcribe_whisperx_disfluent(
    audio_path=os.path.join(DATA_DIR,case_id,audio_filename),
)

2026-08-26 14:04:28 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../opt/homebrew/Caskroom/miniforge/base/envs/md-speech/lib/python3.11/site-packages/whisperx/assets/pytorch_model.bin`


2026-08-26 14:09:22 - whisperx.diarize - INFO - Loading diarization model: pyannote/speaker-diarization-community-1


In [20]:
out_path = os.path.join("../data/transcripts", f'{case_id}_whisperx_disfluent.json')
with open(out_path, "w") as f:
    json.dump(result, f, indent=4)
print(f"Wrote {out_path} ")

Wrote ../data/transcripts/261456_whisperx_disfluent.json 


In [17]:
result.keys()

dict_keys(['segments', 'word_segments'])

[' 15 seconds until the patient encounter begins.',
 'Learners, log in to Learning Space and follow the door-note instructions.',
 " That's what we want.",
 "I'm sorry, everyone.",
 "I'm working about to turn my walkie-talkie off.",
 'I have to hear you.',
 ' Okay.',
 'And just for documentation purposes, uh, should I play He-Him?',
 'He-Him is fine.',
 'He-Him is fine?',
 'Okay.',
 "I'll do that.",
 'Okay.',
 "So, um, I have here on my note that you're here about, um, um, is that true alcohol overuse?",
 'Uh, well, judge told me I need to come in here and talk with you about my drinking.',
 'Yeah, okay.',
 'Okay.',
 ' Could you tell me a little bit about the past history about drinking?',
 'Yeah.',
 'What are you interested in knowing?',
 'Um, roughly how long ago did you first start, uh, drinking?',
 'I — I started when I was a teenager.',
 'Uh-huh.',
 'Okay.',
 'And just to confirm, how old are you right now?',
 "I'm 67.",
 '67.',
 'Okay.',
 'Um, and how much — how many drinks do yo

In [19]:
old = json.load(open(out_path))
[old["segments"][i]["text"] for i in range(len(old["segments"]))]

[' 15 seconds until the patient encounter begins.',
 'Learners, log in to learning space and follow the door note instructions.',
 " I'm sorry, everyone.",
 "I'm worried about the hearing.",
 "I won't be able to hear you now.",
 ' Okay, and just for documentation purposes, should I put he, him?',
 'He, him is fine.',
 'He, him is fine?',
 "Okay, I'll do that.",
 "Okay, so I have here on my note that you're here about, is that true alcohol overuse?",
 'Well, judge told me I need to come in here and talk with you about my drinking.',
 'Yeah, okay, okay.',
 'So...',
 ' Could you tell me a little bit about the past history about drinking?',
 'Yeah, what are you interested in knowing?',
 'Roughly how long ago did you first start drinking?',
 'I started when I was a teenager.',
 'Okay, and just to confirm, how old are you right now?',
 "I'm 67.",
 '67, okay.',
 'And how many drinks do you have per day?',
 " Usually when I get home from work, I'll have two or three beers.",
 "My job is pretty